# 17장 깊게 쌓기 — VGG와 ResNet

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch17_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `vgg.py`, `resnet.py`, `p01_depth_experiment.py` 를 치고 실행 | [ch17_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch17_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `BasicBlock.forward` 세 줄과 `shortcut` 조건문에 "모양이 어떻게 되는가"를 주석으로 | [ch17_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch17_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `BasicBlock` 복원. 그다음 14장 패키지 `networks/` 에 `vgg.py`·`resnet.py` 를 넣고 `--model resnet` 으로 학습되게 등록 | [ch17_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch17_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "17"))                      # 이 장의 코드 폴더로 이동
if not os.path.isdir(os.path.join(os.environ["JUCOS_DATA"], "shapes")):   # 15장의 도형 데이터가 없으면 만든다
    !python ../15/make_shapes.py
print("준비 끝. 작업 폴더: code/17")

준비 끝. 작업 폴더: code/17


## `vgg.py` — 17.1 VGG: 같은 모양의 블록(3x3 합성곱 몇 개 + 풀링)을 반복해 깊게 쌓는다

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 17.1 VGG: 같은 모양의 블록(3x3 합성곱 몇 개 + 풀링)을 반복해 깊게 쌓는다
import torch
import torch.nn as nn


def vgg_block(in_ch, out_ch, n_convs):
    """3x3 합성곱 n_convs 개 (크기 유지) + 2x2 풀링 (크기 절반)"""
    layers = []
    for i in range(n_convs):
        layers += [nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, kernel_size=3, padding=1),
                   nn.BatchNorm2d(out_ch),          # 각 채널의 값을 정규화 → 깊어져도 학습이 안정
                   nn.ReLU(inplace=True)]
    layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)                    # 층들을 순서대로 이어 붙인 하나의 모듈


# 블록 구성표: (출력 채널, 합성곱 수). 원 논문의 A(11층)·D(16층) 구성을 32px 입력에 맞게 줄인 것
CONFIGS = {
    "tiny":  [(16, 1), (32, 1), (64, 1)],                     # 32 → 16 → 8 → 4
    "small": [(32, 2), (64, 2), (128, 2)],
    "vgg11": [(64, 1), (128, 1), (256, 2), (512, 2), (512, 2)],   # 32 → 1
}


class VGG(nn.Module):
    def __init__(self, config="small", in_ch=3, num_classes=10, img_size=32):
        super().__init__()
        blocks, ch = [], in_ch
        for out_ch, n in CONFIGS[config]:
            blocks.append(vgg_block(ch, out_ch, n))
            ch = out_ch
        self.features = nn.Sequential(*blocks)
        side = img_size // (2 ** len(CONFIGS[config]))          # 블록 수만큼 절반
        self.classifier = nn.Sequential(
            nn.Flatten(1),
            nn.Linear(ch * side * side, 256), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


if __name__ == "__main__":
    for name in CONFIGS:
        m = VGG(name)
        n = sum(p.numel() for p in m.parameters())
        out = m(torch.zeros(2, 3, 32, 32))
        print(f"VGG-{name:<6} 블록 {len(CONFIGS[name])}  파라미터 {n:>10,}  [2,3,32,32] → {tuple(out.shape)}")
    print(VGG("tiny"))
```

In [ ]:
!python vgg.py

VGG-tiny   블록 3  파라미터    288,778  [2,3,32,32] → (2, 10)
VGG-small  블록 3  파라미터    815,018  [2,3,32,32] → (2, 10)
VGG-vgg11  블록 5  파라미터  9,359,882  [2,3,32,32] → (2, 10)
VGG(
  (features): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=T

## `resnet.py` — 17.2 ResNet: 블록의 입력을 출력에 더하는 "지름길(skip connection)" 로 아주 깊게 쌓아도 학습되게 한다

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 17.2 ResNet: 블록의 입력을 출력에 더하는 "지름길(skip connection)" 로 아주 깊게 쌓아도 학습되게 한다
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    """3x3 합성곱 둘 + 지름길. out = relu( F(x) + x )"""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        # 지름길: 모양(채널·크기)이 바뀌는 블록에서는 1x1 합성곱으로 x 의 모양을 맞춘다
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                                          nn.BatchNorm2d(out_ch))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))        # 더하기 한 줄이 ResNet 의 전부


class ResNet(nn.Module):
    """입력부 → 단계(stage) 몇 개 → 출력부. 단계마다 채널은 두 배, 크기는 절반."""

    def __init__(self, blocks_per_stage=(2, 2, 2), channels=(16, 32, 64), in_ch=3, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(in_ch, channels[0], 3, padding=1, bias=False),
                                  nn.BatchNorm2d(channels[0]), nn.ReLU(inplace=True))
        stages, ch = [], channels[0]
        for i, (n, out_ch) in enumerate(zip(blocks_per_stage, channels)):
            stride = 1 if i == 0 else 2                       # 첫 단계 외에는 크기 절반
            layers = [BasicBlock(ch, out_ch, stride)] + [BasicBlock(out_ch, out_ch) for _ in range(n - 1)]
            stages.append(nn.Sequential(*layers))
            ch = out_ch
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(1), nn.Linear(ch, num_classes))

    def forward(self, x):
        return self.head(self.stages(self.stem(x)))


if __name__ == "__main__":
    for cfg in [(1, 1, 1), (2, 2, 2), (3, 3, 3)]:
        m = ResNet(cfg)
        n = sum(p.numel() for p in m.parameters())
        depth = 1 + 2 * sum(cfg) + 1
        print(f"ResNet 블록 {cfg}  합성곱 층 수 약 {depth:>2}  파라미터 {n:>9,}  [2,3,32,32] → {tuple(m(torch.zeros(2, 3, 32, 32)).shape)}")
    # 지름길이 모양을 맞추는지
    b = BasicBlock(16, 32, stride=2)
    print("BasicBlock(16→32, stride 2): [2,16,32,32] →", tuple(b(torch.zeros(2, 16, 32, 32)).shape))
```

In [ ]:
!python resnet.py

ResNet 블록 (1, 1, 1)  합성곱 층 수 약  8  파라미터    78,042  [2,3,32,32] → (2, 10)
ResNet 블록 (2, 2, 2)  합성곱 층 수 약 14  파라미터   175,258  [2,3,32,32] → (2, 10)
ResNet 블록 (3, 3, 3)  합성곱 층 수 약 20  파라미터   272,474  [2,3,32,32] → (2, 10)
BasicBlock(16→32, stride 2): [2,16,32,32] → (2, 32, 16, 16)


## `p01_depth_experiment.py` — 17.3 깊이 실험: 지름길이 없으면 깊은 망이 왜 안 배우는지 본다 (도형 데이터, 짧은 학습)

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 17.3 깊이 실험: 지름길이 없으면 깊은 망이 왜 안 배우는지 본다 (도형 데이터, 짧은 학습)
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
import sys
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "15"))
from p02_custom_dataset import ImageFolderDataset       # 15장의 Dataset 재사용

from vgg import VGG
from resnet import ResNet, BasicBlock

ROOT = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "shapes")
tf = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor(), transforms.Normalize((0.5,) * 3, (0.5,) * 3)])
train_loader = DataLoader(ImageFolderDataset(os.path.join(ROOT, "train"), tf), batch_size=32, shuffle=True)
test_loader = DataLoader(ImageFolderDataset(os.path.join(ROOT, "test"), tf), batch_size=64)


class PlainDeep(nn.Module):
    """지름길 없는 깊은 망: 3x3 합성곱 n 개를 그냥 쌓는다 (BatchNorm 도 없이)"""

    def __init__(self, n_layers, ch=16, num_classes=3):
        super().__init__()
        layers = [nn.Conv2d(3, ch, 3, padding=1), nn.ReLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Conv2d(ch, ch, 3, padding=1), nn.ReLU()]
        self.body = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(1), nn.Linear(ch, num_classes))

    def forward(self, x):
        return self.head(self.body(x))


def accuracy(model):
    model.eval()
    c = 0
    with torch.no_grad():
        for x, y in test_loader:
            c += (model(x).argmax(1) == y).sum().item()
    model.train()
    return c / len(test_loader.dataset)


def run(name, model, epochs=6, lr=0.05):
    torch.manual_seed(0)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()
    t0 = time.time()
    for _ in range(epochs):
        for x, y in train_loader:
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            opt.step()
    n = sum(p.numel() for p in model.parameters())
    print(f"{name:<26} 파라미터 {n:>9,}  정확도 {accuracy(model):.3f}  ({time.time() - t0:.0f}초)")


if __name__ == "__main__":
    torch.manual_seed(0); run("Plain 4층", PlainDeep(4))
    torch.manual_seed(0); run("Plain 20층 (지름길 없음)", PlainDeep(20))
    torch.manual_seed(0); run("VGG-tiny (BN 있음)", VGG("tiny", num_classes=3))
    torch.manual_seed(0); run("ResNet (1,1,1)", ResNet((1, 1, 1), num_classes=3))
    torch.manual_seed(0); run("ResNet (3,3,3) 약 20층", ResNet((3, 3, 3), num_classes=3))
```

In [ ]:
!python p01_depth_experiment.py

Plain 4층                   파라미터     7,459  정확도 0.333  (4초)
Plain 20층 (지름길 없음)         파라미터    44,579  정확도 0.333  (13초)
VGG-tiny (BN 있음)           파라미터   286,979  정확도 0.940  (4초)
ResNet (1,1,1)             파라미터    77,587  정확도 0.867  (9초)
ResNet (3,3,3) 약 20층       파라미터   272,019  정확도 1.000  (13초)


## 다음 단계

- 2단계 「주석 달기」: [ch17_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch17_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch17_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch17_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.